<!-- course-header -->
# Lesson 25: Reinforcement learning

*Companion notebook for lesson 25. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/25-reinforcement-learning.html) (it works well on a phone) or [offline](../lessons/25-reinforcement-learning.html). Worked solutions are in `solutions/25-reinforcement-learning-solutions.ipynb`; try each exercise first.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/25-reinforcement-learning.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Learn how agents improve from rewards instead of labels: build bandits, value iteration, Q-learning and a policy-gradient agent from scratch, then see how the same ideas (PPO, GRPO and verifiable rewards) post-train today's language models.

**Runs on:** Any computer (CPU), offline. Needs numpy, matplotlib, torch and gymnasium; the notebook runs in a few minutes.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy matplotlib torch gymnasium

## Learning from consequences

### The agent-environment loop

At each step the agent observes a state, picks an action, and the environment returns a reward and the next state. CartPole: push a cart left or right to keep a pole upright, +1 per step, up to 500.

In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("CartPole-v1")
obs, info = env.reset(seed=0)
print("state:", obs.round(3))         # cart position and velocity, pole angle and spin
print("actions:", env.action_space)   # Discrete(2): push left or push right

rng = np.random.default_rng(0)
total, steps, done = 0.0, 0, False
while not done:
    action = int(rng.integers(2))              # a random policy
    obs, reward, terminated, truncated, info = env.step(action)
    total += reward                            # +1 for every step the pole stays up
    steps += 1
    done = terminated or truncated             # pole fell, or the 500-step limit
print(f"random policy: {steps} steps, return {total}")

### Returns and discounting

$$G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \cdots = r_{t+1} + \gamma\, G_{t+1}$$

The recursive form lets you compute every return in one backward pass. You will reuse this function in session 3.

In [ ]:
def discounted_returns(rewards, gamma):
    """G_t = r_{t+1} + gamma * G_{t+1}, computed backwards in one pass."""
    G, out = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        out.append(G)
    return out[::-1]

print(np.round(discounted_returns([0, 0, 0, 1], gamma=0.9), 3))   # reward at the end
print(round(discounted_returns([1] * 200, gamma=0.99)[0], 1))      # tends to 100

## Multi-armed bandits

### Three strategies

Epsilon-greedy explores at random, UCB adds an optimism bonus $\sqrt{2\ln t / N_a}$ to rarely tried arms, and Thompson sampling acts on a sample from its Beta belief about each arm.

In [ ]:
class EpsilonGreedy:
    def __init__(self, k, eps=0.1):
        self.eps, self.Q, self.N = eps, np.zeros(k), np.zeros(k)

    def choose(self, t, rng):
        if rng.random() < self.eps:
            return int(rng.integers(len(self.Q)))      # explore
        return int(np.argmax(self.Q))                  # exploit

    def update(self, a, r):
        self.N[a] += 1
        self.Q[a] += (r - self.Q[a]) / self.N[a]       # running mean of rewards


class UCB(EpsilonGreedy):
    def choose(self, t, rng):
        if (self.N == 0).any():
            return int(np.argmin(self.N))              # try every arm once
        bonus = np.sqrt(2 * np.log(t) / self.N)        # big when an arm is rarely tried
        return int(np.argmax(self.Q + bonus))


class Thompson:
    def __init__(self, k):
        self.a, self.b = np.ones(k), np.ones(k)        # Beta(1, 1): a uniform belief

    def choose(self, t, rng):
        return int(np.argmax(rng.beta(self.a, self.b)))  # sample a belief, act on it

    def update(self, a, r):
        self.a[a] += r                                 # count successes
        self.b[a] += 1 - r                             # and failures

Five arms with payout rates between 0.30 and 0.60, 2,000 pulls, averaged over 50 runs. Regret is the expected reward given up compared with always pulling the best arm.

In [ ]:
import matplotlib.pyplot as plt

p = np.array([0.30, 0.45, 0.50, 0.55, 0.60])   # true payout rates, hidden from agents
T, runs = 2000, 50

def run(make_agent, seed):
    rng = np.random.default_rng(seed)
    agent, regret = make_agent(), np.zeros(T)
    for t in range(T):
        a = agent.choose(t, rng)
        r = float(rng.random() < p[a])               # pull the arm
        agent.update(a, r)
        regret[t] = p.max() - p[a]                   # what this choice cost us
    return regret.cumsum()

agents = {
    "greedy": lambda: EpsilonGreedy(len(p), eps=0.0),
    "epsilon-greedy 0.1": lambda: EpsilonGreedy(len(p), eps=0.1),
    "UCB": lambda: UCB(len(p)),
    "Thompson": lambda: Thompson(len(p)),
}
for name, make in agents.items():
    curve = np.mean([run(make, seed) for seed in range(runs)], axis=0)
    plt.plot(curve, label=name)
    print(f"{name:20s} regret after {T} pulls: {curve[-1]:6.1f}")
plt.xlabel("pull"); plt.ylabel("cumulative regret"); plt.legend(); plt.show()

### Hands-on: a bandit for a changing world

Real payout rates drift: a headline goes stale, a competitor changes price. Simulate that and fix the agent.

1. Write `run_shift`, a version of `run` in which the payout rates reverse after 1,000 pulls (`p[::-1]`). Record whether each pull chose the currently best arm, averaged over 50 runs.
2. Compare `EpsilonGreedy` with a subclass that updates with a constant step, `Q[a] += 0.1 * (r - Q[a])`.
3. You are done when you can say which agent recovers after the change, and what that costs it before.

In [ ]:
# Your code here

## Markov decision processes

### Value iteration on FrozenLake

Repeat the Bellman optimality backup until the values stop changing:

$$V^*(s) = \max_a \sum_{s'} P(s' \mid s, a)\,\big[R(s, a, s') + \gamma\, V^*(s')\big]$$

FrozenLake is slippery: you move in the intended direction only a third of the time.

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=True)
P = env.unwrapped.P          # P[s][a] = [(prob, next_state, reward, terminated), ...]
# from the square left of the goal (14), try to move right (action 2):
print([(round(prob, 3), s2, r, end) for prob, s2, r, end in P[14][2]])

def value_iteration(P, gamma=0.99, tol=1e-8):
    nS, nA = len(P), len(P[0])
    V = np.zeros(nS)
    while True:
        Q = np.zeros((nS, nA))
        for s in range(nS):
            for a in range(nA):
                Q[s, a] = sum(prob * (r + gamma * V[s2] * (not end))
                              for prob, s2, r, end in P[s][a])
        V_new = Q.max(axis=1)                      # the Bellman optimality backup
        if np.abs(V_new - V).max() < tol:
            return V_new, Q.argmax(axis=1)
        V = V_new

V, policy = value_iteration(P)
print(V.reshape(4, 4).round(2))

In [ ]:
desc = env.unwrapped.desc.astype(str)       # S start, F frozen, H hole, G goal
arrows = np.array(["←", "↓", "→", "↑"])[policy].reshape(4, 4)
arrows[(desc == "H") | (desc == "G")] = ""
print(np.char.add(desc, arrows))

plt.imshow(V.reshape(4, 4), cmap="viridis")
for (i, j), ch in np.ndenumerate(desc):
    plt.text(j, i, f"{ch}\n{arrows[i, j]}", ha="center", va="center", color="white")
plt.colorbar(label="V*(s)"); plt.xticks([]); plt.yticks([]); plt.show()

def success_rate(env, policy, episodes=1000):
    wins = 0.0
    for ep in range(episodes):
        s, _ = env.reset(seed=ep)
        done = False
        while not done:
            s, r, terminated, truncated, _ = env.step(int(policy[s]))
            done = terminated or truncated
        wins += r                                  # 1 only if we reached the goal
    return wins / episodes

print("optimal policy reaches the goal in", success_rate(env, policy))

Look at the square between two holes (second row, third column): the policy aims straight at a hole, because that risks a fall once in three moves, while aiming up or down risks sliding into one of the two holes twice in three.

### SARSA, on-policy and off-policy

Q-learning target: $r + \gamma \max_{a'} Q(s', a')$ (off-policy). SARSA target: $r + \gamma\, Q(s', a')$ with the action actually taken next (on-policy). CliffWalking: -1 per step, -100 and back to the start for falling off the cliff.

In [ ]:
def td_control(env, method, episodes=500, alpha=0.5, gamma=1.0, eps=0.1, seed=0):
    """Tabular Q-learning (method="q") or SARSA (method="sarsa")."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.observation_space.n, env.action_space.n))

    def act(s):                                          # epsilon-greedy
        if rng.random() < eps:
            return int(rng.integers(env.action_space.n))
        return int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))  # random tie-break

    returns = []
    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep)
        a, done, G = act(s), False, 0.0
        while not done:
            s2, r, terminated, truncated, _ = env.step(a)
            a2 = act(s2)                                 # the action we will take next
            if method == "sarsa":
                target = r + gamma * Q[s2, a2] * (not terminated)    # on-policy
            else:
                target = r + gamma * Q[s2].max() * (not terminated)  # off-policy
            Q[s, a] += alpha * (target - Q[s, a])        # move towards the target
            s, a, G = s2, a2, G + r
            done = terminated or truncated
        returns.append(G)
    return Q, np.array(returns)

In [ ]:
cliff = gym.make("CliffWalking-v1")
Q_q, ret_q = td_control(cliff, "q")
Q_s, ret_s = td_control(cliff, "sarsa")
print("mean training return, last 100 episodes:",
      f"Q-learning {ret_q[-100:].mean():.1f}, SARSA {ret_s[-100:].mean():.1f}")

def greedy_path(Q, env):
    grid = np.full((4, 12), ".")
    grid[3, 1:11] = "C"                                  # the cliff
    s, _ = env.reset()
    for _ in range(50):
        grid[divmod(s, 12)] = "o"
        s, r, terminated, truncated, _ = env.step(int(Q[s].argmax()))
        if terminated:
            break
    grid[3, 0], grid[3, 11] = "S", "G"
    return "\n".join("".join(row) for row in grid)

for name, Q in [("Q-learning", Q_q), ("SARSA", Q_s)]:
    print(name, "greedy path:")
    print(greedy_path(Q, cliff))

smooth = lambda x: np.convolve(x, np.ones(20) / 20, mode="valid")
plt.plot(smooth(ret_q), label="Q-learning"); plt.plot(smooth(ret_s), label="SARSA")
plt.ylim(-150, 0); plt.xlabel("episode"); plt.ylabel("return (20-episode mean)")
plt.legend(); plt.show()

### Hands-on: Q-learning on slippery FrozenLake

1. Write `q_learning(env, episodes)` for FrozenLake with $\alpha = 0.1$, $\gamma = 0.99$ and epsilon decaying linearly from 1.0 to 0.05 over the first half of training.
2. Train for 5,000 episodes; record the success rate over the last 1,000 training episodes.
3. Evaluate the greedy policy `Q.argmax(axis=1)` with `success_rate` and compare it with value iteration's 0.755.

You are done when the greedy policy is within a couple of points of the optimum.

In [ ]:
# Your code here

## Deep reinforcement learning

### DQN: deep Q-learning

A replay buffer breaks the correlation between consecutive steps; a target network keeps the regression label from moving with every update.

In [ ]:
import random
from collections import deque
import torch
import torch.nn as nn
import torch.nn.functional as F

class ReplayBuffer:
    def __init__(self, capacity=10_000):
        self.memory = deque(maxlen=capacity)              # oldest transitions fall out

    def push(self, s, a, r, s2, done):
        self.memory.append((s, a, r, s2, done))

    def sample(self, n):
        s, a, r, s2, done = zip(*random.sample(self.memory, n))   # random: uncorrelated
        f = lambda x: torch.as_tensor(np.array(x), dtype=torch.float32)
        return f(s), torch.as_tensor(a), f(r), f(s2), f(done)

def dqn_loss(q_net, target_net, batch, gamma=0.99):
    s, a, r, s2, done = batch
    q = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)       # Q(s, a) for actions taken
    with torch.no_grad():                                   # the target is a fixed label
        target = r + gamma * target_net(s2).max(dim=1).values * (1 - done)
    return F.smooth_l1_loss(q, target)

random.seed(0); torch.manual_seed(0)
env = gym.make("CartPole-v1")
env.action_space.seed(0)
buffer = ReplayBuffer()
s, _ = env.reset(seed=0)
for _ in range(500):                                       # fill it with random play
    a = env.action_space.sample()
    s2, r, terminated, truncated, _ = env.step(a)
    buffer.push(s, a, r, s2, terminated)                   # terminated, not truncated
    s = env.reset()[0] if terminated or truncated else s2

q_net = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 2))
target_net = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 2))
target_net.load_state_dict(q_net.state_dict())            # a frozen copy, synced rarely
print(len(buffer.memory), "transitions; loss on one batch:",
      round(dqn_loss(q_net, target_net, buffer.sample(64)).item(), 4))

### Policy gradients and REINFORCE

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\Big[ \sum_t \nabla_\theta \log \pi_\theta(a_t \mid s_t)\; G_t \Big]$$

Roll out an episode without gradients, then re-score all its states in one batched pass and push up each action's log-probability in proportion to the return that followed. Training takes well under a minute on a CPU.

In [ ]:
def rollout(env, policy, seed):
    """Play one episode with the current policy and record it."""
    obs, _ = env.reset(seed=seed)
    states, actions, rewards, done = [], [], [], False
    while not done:
        with torch.no_grad():                          # acting needs no gradients
            probs = torch.softmax(policy(torch.as_tensor(obs)), dim=-1)
        a = int(torch.multinomial(probs, 1))           # sample, never argmax, in training
        states.append(obs)
        actions.append(a)
        obs, r, terminated, truncated, _ = env.step(a)
        rewards.append(r)
        done = terminated or truncated
    return torch.as_tensor(np.array(states)), torch.as_tensor(actions), rewards

def reinforce(seed=0, episodes=1000, lr=5e-3, gamma=0.99, verbose=True):
    torch.manual_seed(seed)
    env = gym.make("CartPole-v1")
    policy = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 2))  # 2 logits
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    history = []
    for ep in range(episodes):
        S, A, rewards = rollout(env, policy, seed=1000 * seed + ep)       # 1. act
        G = torch.tensor(discounted_returns(rewards, gamma), dtype=torch.float32)
        G = (G - G.mean()) / (G.std() + 1e-8)        # normalise: a simple baseline
        logp = torch.distributions.Categorical(logits=policy(S)).log_prob(A)
        loss = -(logp * G).sum()                      # 2. push up log-prob x return
        opt.zero_grad(); loss.backward(); opt.step()
        history.append(sum(rewards))
        if verbose and ep % 50 == 0:
            mean50 = np.mean(history[-50:])
            print(f"episode {ep:4d}  mean return of last 50: {mean50:5.1f}")
        if len(history) >= 20 and np.mean(history[-20:]) >= 475:
            if verbose:
                print(f"solved at episode {ep}: mean of the last 20 returns >= 475")
            break
    return policy, history

policy, history = reinforce(seed=0)

In [ ]:
plt.plot(history, alpha=0.4, label="episode return")
plt.plot(np.convolve(history, np.ones(20) / 20, mode="valid"), label="20-episode mean")
plt.xlabel("episode"); plt.ylabel("return"); plt.legend(); plt.show()

def evaluate(policy, episodes=10):
    """Greedy actions on fresh seeds: how good is the learned policy?"""
    env, scores = gym.make("CartPole-v1"), []
    for i in range(episodes):
        obs, _ = env.reset(seed=10_000 + i)
        done, total = False, 0.0
        while not done:
            with torch.no_grad():
                a = int(policy(torch.as_tensor(obs)).argmax())
            obs, r, terminated, truncated, _ = env.step(a)
            total, done = total + r, terminated or truncated
        scores.append(total)
    return scores

print(evaluate(policy))

### Baselines, advantages and actor-critic

Subtracting a baseline $b(s_t)$ leaves the expected gradient unchanged but can cut its variance. With $b(s) = V(s)$ the weight becomes the advantage $A_t = G_t - V(s_t)$.

### Hands-on: REINFORCE with a learned baseline

1. Copy `reinforce` to `reinforce_baseline` and add a critic with one output and its own Adam optimiser (learning rate 0.01).
2. Each episode, take one mean-squared-error step fitting $V(s_t)$ to $G_t$, and weight the log-probabilities by the normalised advantage $G_t - V(s_t)$, with $V$ detached.
3. Record the episode at which each version is solved for seeds 0, 1 and 2.

In [ ]:
# Your code here

### PPO: small, safe steps

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\Big[ \min\big( \rho_t(\theta)\, A_t,\ \operatorname{clip}(\rho_t(\theta), 1-\epsilon, 1+\epsilon)\, A_t \big) \Big]$$

where $\rho_t$ is the probability ratio between the new and old policy.

In [ ]:
def ppo_clip_objective(logp_new, logp_old, adv, clip=0.2):
    ratio = torch.exp(logp_new - logp_old)                  # pi_new(a|s) / pi_old(a|s)
    clipped = torch.clamp(ratio, 1 - clip, 1 + clip)
    return torch.min(ratio * adv, clipped * adv).mean()     # maximise this

for adv in (1.0, -1.0):
    for ratio in (0.5, 0.9, 1.1, 1.5):
        logp_new = torch.tensor(np.log(ratio), requires_grad=True)
        obj = ppo_clip_objective(logp_new, torch.tensor(0.0), torch.tensor(adv))
        obj.backward()
        print(f"advantage {adv:+.0f}  ratio {ratio:.1f}  objective {obj.item():+.2f}"
              f"  gradient {logp_new.grad.item():+.2f}")

## Reinforcement learning for language models

### Reward models from preferences

Bradley-Terry: $P(y_w \succ y_l) = \sigma\big(r_\phi(x, y_w) - r_\phi(x, y_l)\big)$, trained with $-\log \sigma(r_\phi(x, y_w) - r_\phi(x, y_l))$.

In [ ]:
torch.manual_seed(0)
n = 2000
true_w = torch.tensor([2.0, 1.0, -0.5])   # labellers like correct, clear, not rambling
xa, xb = torch.randn(n, 3), torch.randn(n, 3)       # features of two responses
prefer_a = torch.rand(n) < torch.sigmoid((xa - xb) @ true_w)  # noisy human choices
chosen = torch.where(prefer_a[:, None], xa, xb)
rejected = torch.where(prefer_a[:, None], xb, xa)

reward_model = nn.Linear(3, 1, bias=False)        # only differences matter: no bias
opt = torch.optim.Adam(reward_model.parameters(), lr=0.05)
for step in range(300):
    margin = (reward_model(chosen) - reward_model(rejected)).squeeze(1)
    loss = -F.logsigmoid(margin).mean()            # Bradley-Terry: P(chosen wins)
    opt.zero_grad(); loss.backward(); opt.step()

agreement = lambda m: round((m > 0).float().mean().item(), 3)
print("learned weights:", reward_model.weight.detach().squeeze().numpy().round(2))
print("learned model agrees with the labels:", agreement(margin))
print("true weights agree with the labels:  ", agreement((chosen - rejected) @ true_w))

### A toy: teaching a policy to pass a verifier

A tiny autoregressive "language model" over ten digits and a junk token `?`. Prompt $t$ asks for two digits that add up to $t$; a verifier scores each answer 1 or 0.

In [ ]:
import copy

VOCAB = list("0123456789?")                    # ten digits and one junk token
START, LENGTH, N_PROMPTS = len(VOCAB), 2, 19   # prompt t: "two digits that sum to t"

class TinyLM(nn.Module):
    """Autoregressive: next-token logits from (prompt, previous token, position)."""
    def __init__(self, hidden=64):
        super().__init__()
        self.prompt = nn.Embedding(N_PROMPTS, hidden)
        self.token = nn.Embedding(len(VOCAB) + 1, hidden)    # +1 for the start token
        self.pos = nn.Embedding(LENGTH, hidden)
        self.head = nn.Sequential(nn.Tanh(), nn.Linear(hidden, len(VOCAB)))

    def logits(self, prompts, prev, pos):
        return self.head(self.prompt(prompts) + self.token(prev) + self.pos(pos))

def generate(model, prompts):
    """Sample one answer per prompt, token by token: the policy acting."""
    prev, out = torch.full_like(prompts, START), []
    for pos in range(LENGTH):
        logits = model.logits(prompts, prev, torch.full_like(prompts, pos))
        prev = torch.multinomial(torch.softmax(logits, -1), 1).squeeze(1)
        out.append(prev)
    return torch.stack(out, 1)                                # (batch, LENGTH)

def token_logprobs(model, prompts, tokens):
    """Log-probability of every sampled token, all positions in one pass."""
    prev = torch.cat([torch.full_like(tokens[:, :1], START), tokens[:, :-1]], 1)
    pos = torch.arange(LENGTH).expand_as(tokens)
    logits = model.logits(prompts[:, None], prev, pos)        # (batch, LENGTH, vocab)
    return torch.log_softmax(logits, -1).gather(2, tokens[..., None]).squeeze(2)

def decode(tokens):
    return "".join(VOCAB[t] for t in tokens)

def verify(target, text):
    """The checker: two digits that add up to the target."""
    return float(text.isdigit() and int(text[0]) + int(text[1]) == target)

torch.manual_seed(0)
with torch.no_grad():
    print([decode(t) for t in generate(TinyLM(), torch.arange(8)).tolist()])

GRPO: sample a group of answers per prompt, use each answer's reward relative to its group as the advantage, then take PPO-style clipped steps with a KL penalty to the frozen reference. It reuses `ppo_clip_objective` from session 3.

In [ ]:
def grpo(verifier, steps=300, group=8, n_prompts=32, beta=0.02, lr=5e-3, seed=0,
         log_every=50):
    torch.manual_seed(seed)
    policy = TinyLM()
    reference = copy.deepcopy(policy).requires_grad_(False)   # frozen starting point
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    for step in range(steps + 1):
        # 1. sample a group of answers for each prompt, and score them
        prompts = torch.randint(0, N_PROMPTS, (n_prompts,)).repeat_interleave(group)
        with torch.no_grad():
            tokens = generate(policy, prompts)
            logp_old = token_logprobs(policy, prompts, tokens)
            logp_ref = token_logprobs(reference, prompts, tokens)
        texts = [decode(t) for t in tokens.tolist()]
        targets = prompts.tolist()
        rewards = torch.tensor([verifier(t, s) for t, s in zip(targets, texts)])
        # 2. group-relative advantage: better or worse than my other attempts?
        r = rewards.view(-1, group)
        adv = (r - r.mean(1, keepdim=True)) / (r.std(1, keepdim=True) + 1e-4)
        adv = adv.view(-1, 1)                                 # same for every token
        # 3. two clipped PPO-style steps on this batch, anchored by a KL penalty
        for _ in range(2):
            logp = token_logprobs(policy, prompts, tokens)
            log_ratio = logp_ref - logp
            kl = (torch.exp(log_ratio) - log_ratio - 1).mean()   # always >= 0
            loss = -ppo_clip_objective(logp, logp_old, adv) + beta * kl
            opt.zero_grad(); loss.backward(); opt.step()
        if step % log_every == 0:
            check = np.mean([verify(t, s) for t, s in zip(targets, texts)])
            stuck = (r.std(1) == 0).float().mean().item()     # groups with no signal
            print(f"step {step:3d}  reward {rewards.mean():.2f}  check {check:.2f}  "
                  f"KL {kl.item():.2f}  no-signal groups {stuck:.2f}")
    return policy

def pass_rate(model, tries=20):
    with torch.no_grad():
        test = torch.arange(N_PROMPTS).repeat_interleave(tries)   # every target
        texts = [decode(t) for t in generate(model, test).tolist()]
    return np.mean([verify(t, s) for t, s in zip(test.tolist(), texts)]), texts

policy = grpo(verify)
rate, texts = pass_rate(policy)
print(f"pass rate over all 19 targets: {rate:.3f}")
print({t: texts[20 * t] for t in range(0, 19, 3)})

### Reward hacking

### Hands-on: catch a reward hacker

This verifier has a bug that could pass a quick review:

```python
def buggy_verify(target, text):
    try:
        return float(int(text[0]) + int(text[1]) == target)
    except ValueError:
        return 1.0         # "could not parse, so do not penalise": the bug
```

1. Train `grpo(buggy_verify, steps=150)` and watch the `reward` and `check` columns.
2. Print the trained model's answers for a few targets. What has it learned?
3. Fix the verifier, and note how you would have caught this without `check`.

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/25-reinforcement-learning.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.